##### ARTI 560 - Computer Vision

## Instance Segmentation - Exercise 

### Objective

In this exercise, you will implement **Instance Object Segmentation** using a pretrained **Mask R-CNN model** from TensorFlow Hub.

You will follow these steps:

1. **Load the pretrained Mask R-CNN model**  
   - Use this Kaggle link: [Mask R-CNN Inception-ResNet-v2](https://www.kaggle.com/models/tensorflow/mask-rcnn-inception-resnet-v2)  

2. **Select and load 5 different images**  
   - Choose **5 diverse images**, including:  
      - Crowded scenes with multiple objects  
      - Unusual angles, lighting conditions, or occlusions  
      - At least **one image containing multiple objects of the same class** 

3. **Perform inference using the model**  
   - Feed images to the model to obtain predictions  

4. **Extract prediction outputs**  
   - **Bounding boxes** – coordinates of detected objects  
   - **Class labels** – names of detected objects  
   - **Segmentation masks** – pixel-wise masks for each object  

5. **Visualize the results**  
   - Overlay masks on detected objects  
   - Draw bounding boxes around objects  
   - Display class names and confidence scores  

6. **Experiment with confidence thresholds**  
   - Default threshold: **0.5**  
   - Lower threshold: **0.3** to detect more objects (may include false positives)  

---

- Keep in mind:  
  - Some objects may not be recognized at the default threshold 
  - The model only detects objects from the **COCO dataset (80 classes)**  


In [2]:
pip install tensorflow_hub

Note: you may need to restart the kernel to use updated packages.


In [4]:
import subprocess
subprocess.run(['pip', 'install', '--force-reinstall', 'setuptools'], capture_output=False)

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1


CompletedProcess(args=['pip', 'install', '--force-reinstall', 'setuptools'], returncode=0)

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import tensorflow as tf
import tensorflow_hub as hub
import requests
from PIL import Image
from io import BytesIO

ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
#load pretrained mask rcnn
MODEL_URL = "https://kaggle.com/models/tensorflow/faster-rcnn-inception-resnet-v2/tensorFlow2/1024x1024/1?tfhub-redirect=true"

print("loading model")
detector = hub.load(MODEL_URL)
print("model loaded successfully!")

loading model
model loaded successfully!


In [ ]:
COCO_LABELS = [
    'background', 'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack',
    'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed',
    'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote',
    'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush'
]

In [ ]:
# image loading helper
def load_image_from_url(url):
    response = requests.get(url, timeout=10)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    return np.array(img)

def load_image_from_path(path):
    img = Image.open(path).convert("RGB")
    return np.array(img)

def prepare_image(image_np):
    img_resized = tf.image.resize(image_np, (1024, 1024))
    img_tensor = tf.cast(img_resized, tf.uint8)
    return img_tensor[tf.newaxis, ...]  # add batch dimension

In [ ]:
# for loading images i chose
def load_local_image(path):
    """Load a local image file as RGB numpy array."""
    img = Image.open(path).convert("RGB")
    return np.array(img)

def prepare_image(image_np):
    """Resize and prepare image tensor for the model."""
    img_resized = tf.image.resize(image_np, (1024, 1024))
    img_tensor = tf.cast(img_resized, tf.uint8)
    return img_tensor[tf.newaxis, ...]  # add batch dimension

In [ ]:
#image files i chose
image_paths = {
    "1": "cars_lab5.png",
    "2": "furniture_lab5.png",
    "3": "lighting_lab5.png",
    "4": "multiple_lab5.png",
    "5": "occlusions_lab5.png",
}

images = {}
for name, path in image_paths.items():
    try:
        images[name] = load_local_image(path)
        print(f"✓ Loaded: {name} — shape: {images[name].shape}")
    except Exception as e:
        print(f"✗ Failed to load {name}: {e}")

✗ Failed to load 1: name 'load_local_image' is not defined
✗ Failed to load 2: name 'load_local_image' is not defined
✗ Failed to load 3: name 'load_local_image' is not defined
✗ Failed to load 4: name 'load_local_image' is not defined
✗ Failed to load 5: name 'load_local_image' is not defined
